# Detection Sanity Check

Verify that the pretrained **YOLO11n** (COCO weights) loads correctly and produces sensible detections on a reference image before integrating it into the DOBER pipeline.

**What this notebook validates:**
- Model download / load path
- Inference speed on a single image
- Detection output schema (`class`, `confidence`, `bbox`)

**Test image:** `bus.jpg` (Ultralytics sample — contains 1 bus + multiple pedestrians)

## 1. Setup — Load YOLO11n model

In [1]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
print("Model loaded successfully")

Model loaded successfully


## 2. Run inference on sample image

`.predict()` downloads the image, runs a single forward pass, and returns a `Results` object.

In [2]:
results = model.predict(source = "https://ultralytics.com/images/bus.jpg")
results[0].show()


image 1/1 c:\Users\gmission\Desktop\jetson-edge-surveillance\notebooks\bus.jpg: 640x480 4 persons, 1 bus, 40.3ms
Speed: 6.5ms preprocess, 40.3ms inference, 8.3ms postprocess per image at shape (1, 3, 640, 480)


## 3. Inspect detection output

Iterate over `results[0].boxes` to print each detection's class name, confidence, and bounding box.

In [3]:
for box in results[0].boxes:
    name = results[0].names[int(box.cls[0])]
    conf = float(box.conf[0])
    bbox = box.xyxy[0].tolist()
    print(f"{name}: {conf:.2f}, bbox: {bbox}")

bus: 0.94, bbox: [3.8367347717285156, 229.36520385742188, 796.2062377929688, 728.4124145507812]
person: 0.89, bbox: [671.0260009765625, 394.8305969238281, 809.80859375, 878.7166748046875]
person: 0.88, bbox: [47.41636657714844, 399.5693664550781, 239.31118774414062, 904.19677734375]
person: 0.86, bbox: [223.06460571289062, 408.68804931640625, 344.46160888671875, 860.43994140625]
person: 0.62, bbox: [0.02186751365661621, 556.0598754882812, 68.86890411376953, 872.3634643554688]


## 4. Results

| Class  | Count | Confidence range |
|--------|-------|------------------|
| bus    | 1     | 0.94             |
| person | 4     | 0.62 – 0.89      |

- **Inference time:** ~40 ms (laptop GPU baseline)
- **Total pipeline:** ~55 ms (preprocess + inference + postprocess)
- Detection schema verified — ready to consume downstream by `events/` modules (intrusion, loitering, line crossing).

> On Jetson Orin Nano Super the same inference should land in the 15–25 ms range with TensorRT FP16 (measured in a later phase).